In [2]:
import pandas as pd
from pathlib import Path

In [3]:
DATA_PATH = Path("/Users/sricharanreddykonyala/Documents/Projects/sqlguard/data/raw/dataset.csv")

df = pd.read_csv(DATA_PATH)

/var/folders/ln/vfn1t5j955n8mn7gy_x7764w0000gn/T/ipykernel_16016/571577562.py:3: DtypeWarning: Columns (3: attack_stage, 4: tamper_method, 5: attack_status, 8: attack_id, 9: attack_technique) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


In [9]:
df.columns.to_list()

['full_query',
 'label',
 'user_inputs',
 'attack_stage',
 'tamper_method',
 'attack_status',
 'statement_type',
 'query_template_id',
 'attack_id',
 'attack_technique',
 'split']

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3688977 entries, 0 to 3688976
Data columns (total 11 columns):
 #   Column             Dtype
---  ------             -----
 0   full_query         str  
 1   label              int64
 2   user_inputs        str  
 3   attack_stage       str  
 4   tamper_method      str  
 5   attack_status      str  
 6   statement_type     str  
 7   query_template_id  str  
 8   attack_id          str  
 9   attack_technique   str  
 10  split              str  
dtypes: int64(1), str(10)
memory usage: 309.6 MB


In [30]:
df.isnull().sum()

full_query                 0
label                      0
user_inputs            34579
attack_stage         3352696
tamper_method        3353785
attack_status        3352696
statement_type             0
query_template_id       1089
attack_id            3352696
attack_technique     3352696
split                      0
dtype: int64

In [5]:
rows = df.shape[0]
cols = df.shape[1]

print(f"rows: {rows}\ncols: {cols}")

rows: 3688977
cols: 11


In [28]:
query = df[(df['label'] == 1) & (df['attack_status'] == 'success')]['full_query'].iloc[0]

print(query)

INSERT INTO airport (ident, type, name, latitude_deg, longitude_deg) VALUES ("DE-0285", "small_airport", "CAGC Berwick Heliport", 23.90318, 8.52886)


In [27]:
df['label'].unique()

array([1, 0])

In [35]:
# Inspect the provided split values
print("Split values:")
print(df["split"].value_counts(dropna=False))

print("\nLabels by split:")
display(pd.crosstab(df["split"], df["label"], dropna=False))

# Check attack outcomes by split
print("\nAttack status by split:")
display(pd.crosstab(df["split"], df["attack_status"], dropna=False))

Split values:
split
test     3353671
train     335306
Name: count, dtype: int64

Labels by split:


label,0,1
split,,
test,3017390,336281
train,335306,0



Attack status by split:


attack_status,failure,success,NaN
split,,,
test,75607,260674,3017390
train,0,0,335306


In [36]:
print("Duplicate full_query rows:", df["full_query"].duplicated().sum())

print("Queries with conflicting labels:", 
      df.groupby("full_query")["label"].nunique().gt(1).sum()
)

Duplicate full_query rows: 1396058
Queries with conflicting labels: 0


In [37]:
print("Missing template IDs:", df["query_template_id"].isna().sum())

print("\nMost common templates:")
display(df["query_template_id"].value_counts().head(15))

print("\nUnique templates by split:")

display(df.groupby("split")["query_template_id"].nunique())

Missing template IDs: 1089

Most common templates:


query_template_id
airport-S10    112798
airport-S20    112062
airport-S18    111339
airport-S17    110526
airport-S9     110067
airport-S15    109778
airport-S14    109652
airport-S1     109562
airport-S3     109490
airport-S21    109398
airport-S12    109350
airport-S22    106571
airport-S2     106320
airport-S5     103902
airport-S11    103695
Name: count, dtype: int64


Unique templates by split:


split
test     65
train    65
Name: query_template_id, dtype: int64

In [38]:
df["query_length"] = df["full_query"].str.len()

print("Query length statistics:")
display(df["query_length"].describe())

print("\nShortest query:")
print(df.loc[df["query_length"].idxmin(), "full_query"])

print("\nLongest query length:")
print(df["query_length"].max())

Query length statistics:


count    3.688977e+06
mean     1.935497e+02
std      2.413402e+02
min      7.000000e+00
25%      8.100000e+01
50%      1.500000e+02
75%      2.620000e+02
max      5.266000e+03
Name: query_length, dtype: float64


Shortest query:
USE sys

Longest query length:
5266


In [ ]:
print("Query length by label:")
display(df.groupby("label")["query_length"].describe().round(2))

Query length by label:


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,3352696.0,171.00,130.31,7.0,74.0,136.0,201.0,682.0
1,336281.0,418.41,643.44,8.0,201.0,307.0,382.0,5266.0


In [ ]:
print("Query length by split:")
display(df.groupby("split")["query_length"].describe().round(2))

Query length by split:


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,3353671.0,195.80,249.64,7.0,81.0,151.0,263.0,5266.0
train,335306.0,171.02,130.08,7.0,75.0,136.0,201.0,682.0


In [42]:
print("Statement types by label(Proportion):")

display(pd.crosstab(df["statement_type"],df["label"],normalize="columns").round(4))

Statement types by label(Proportion):


label,0,1
statement_type,,
admin,0.0100,0.0000
delete,0.0902,0.0907
insert,0.1002,0.3648
insider,0.0000,0.0032
select,0.6997,0.3178
update,0.0999,0.2234


In [43]:
print("Label distribution by split:")

display(
    pd.crosstab(
        df["split"],
        df["label"],
        margins=True
    )
)

print("\nDuplicate queries by split:")

display(
    df.groupby("split")["full_query"]
      .apply(lambda s: s.duplicated().sum())
)

Label distribution by split:


label,0,1,All
split,,,
test,3017390,336281,3353671
train,335306,0,335306
All,3352696,336281,3688977



Duplicate queries by split:


split
test     1236074
train      90695
Name: full_query, dtype: int64

In [47]:
train_queries = set(
    df.loc[df["split"] == "train", "full_query"]
)

test_queries = set(
    df.loc[df["split"] == "test", "full_query"]
)

shared_queries = train_queries & test_queries

print("Unique train queries:", len(train_queries))
print("Unique test queries:", len(test_queries))
print("Unique queries shared across splits:", len(shared_queries))

Unique train queries: 244611
Unique test queries: 2117597
Unique queries shared across splits: 69289


In [ ]:
# Vectorization

In [48]:
# Select benign queries from the official training split
train_benign = df.loc[(df["split"] == "train") & (df["label"] == 0),"full_query"]

# sampling out of train set
train_sample = train_benign.sample(n=min(30000, len(train_benign)),random_state=42).reset_index(drop=True)

print("Training samples:", len(train_sample))
print("\nSample queries:")
display(train_sample.head())

Training samples: 30000

Sample queries:


0    SELECT icao_code, iata_code, local_code FROM a...
1    SELECT a.id, a.ident, a.name, a.latitude_deg, ...
2    INSERT INTO countries (code, name, continent) ...
3    INSERT INTO countries (code, name, continent, ...
4    SELECT a.name AS airport_name, a.ident AS airp...
Name: full_query, dtype: str

In [ ]:
# tf-idf conversion of queries
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=50_000,
    dtype="float32",
    sublinear_tf=True
)

X_train = tfidf.fit_transform(train_sample)

print("Feature matrix shape:", X_train.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))
print("Matrix type:", type(X_train).__name__)
print("Matrix density:", round(
    X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100, 4
), "%")